# TPWRS PRP Publications by Year

How the IEEE *Transactions on Power Systems* corpus tracks the 2025 Research Agenda, 2015-2026.

Reading notes that apply to every figure below:

- **Every trend is a share of that year's papers, never a count.** TPWRS volume swings from 358 papers
  (2015) to 703 (2018), so raw counts mostly measure how much the journal published.
- **2026 is a partial year** (collected to August 2026) and is shaded in every time series.
- **These are *coverage* measures**: each paper contributes its single best-matching question inside a
  programme, so a figure reads "how many papers this programme reaches", not "how much work each paper
  does". Pooling all abstract-question pairs instead would measure intensity and would be diluted by
  programmes holding many narrow questions.
- Level bands come from `data/prompts/affinity_scoring_scale.yaml`: Low <= 40, Moderate 40-80, High > 80.

## 1. Configuration

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
# import plotly.graph_objects as go
import plotly.io as pio
# from plotly.subplots import make_subplots

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.append(str(repo_root))
from src.analysis import assign_prps

BENCHMARK_ROOT = repo_root / "data" / "results" / "affinity_benchmark"
OUTPUT_DIR = repo_root / "notebooks" / "outputs" / "tpwrs_prp_publications_by_year"
SO_SCREENING_DIR = repo_root / "notebooks" / "input" / "system_operators_IEEE_screening"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROGRAMME_ORDER = ["IBR", "Stability", "DER", "CROF", "System Services", "Planning"]
LEVEL_ORDER = ["Low", "Moderate", "High"]
LEVEL_DTYPE = pd.CategoricalDtype(categories=LEVEL_ORDER, ordered=True)
THRESHOLDS = {"Moderate+": ["Moderate", "High"], "High": ["High"]}
ERA_BINS = [2014, 2017, 2020, 2023, 2026]
# ERA_LABELS = ["2015-17", "2018-20", "2021-23", "2024-26"]
PARTIAL_YEARS = [2026]

# Palettes validated with the dataviz skill's scripts/validate_palette.py --mode light.
# The ordered Low/Moderate/High ramp and its 2-step subset pass --ordinal; the gain/loss pair passes
# --pairs all. The six-programme categorical set is NOT used for line charts: under --pairs all it
# fails (Planning #008300 vs Stability #eb6834 is dE 3.2 under protanopia), so programmes are separated
# by facet position here, not by hue.
LEVEL_COLORS = {"Low": "#86b6ef", "Moderate": "#2a78d6", "High": "#104281"}
BLUE_RAMP = ["#86b6ef", "#4b8fe0", "#2a78d6", "#104281"]
ACCENT, CONTEXT = "#104281", "#898781"
LOSS, GAIN = "#2a78d6", "#e34948"
SURFACE, INK, INK_SECONDARY, INK_MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
PARTIAL_BAND = "#f1f0ea"

latex_textwidth_pt = 516.0 # double column # use this command in latex: \the\textwidth
# latex_textwidth_pt = 452.0 # single column
scale = 1
FIGURE_DIM = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)

FOOTNOTE = (
    "*2026 partial (collected to Aug 2026)."
)
# pio.templates["gpst"] = go.layout.Template(
#     layout=dict(
#         paper_bgcolor=SURFACE,
#         plot_bgcolor=SURFACE,
#         font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", size=12, color=INK_SECONDARY),
#         title=dict(font=dict(size=16, color=INK), x=0, xanchor="left", xref="paper", y=0.97, yanchor="top"),
#         xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor=GRID, linewidth=1,
#                    ticks="outside", ticklen=4, tickcolor=GRID, tickfont=dict(color=INK_MUTED, size=11)),
#         yaxis=dict(showgrid=True, gridcolor=GRID, gridwidth=1, zeroline=False, showline=False,
#                    ticks="", tickfont=dict(color=INK_MUTED, size=11)),
#         legend=dict(title_text="", orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
#                     font=dict(color=INK_SECONDARY, size=11)),
#         margin=dict(l=64, r=40, t=104, b=76),
#     )
# )
# pio.templates.default = "plotly_white"
# SAVE = True



In [2]:
def share_by(assignments, *group_columns, totals):
    """Share of each group's papers in `assignments` that reach each threshold in THRESHOLDS.

    `totals` must be indexed by the same columns that define the denominator group,
    otherwise the share is taken against a wider population than the rows counted.
    """
    join_keys = list(totals.index.names)
    frames = [
        assignments[assignments["Affinity_Level"].isin(levels)]
        .groupby(list(group_columns), observed=True)["Abstract_Index"].nunique()
        .rename("Papers_Reaching").reset_index().assign(Threshold=label)
        for label, levels in THRESHOLDS.items()
    ]
    share = pd.concat(frames, ignore_index=True).merge(totals.reset_index(), on=join_keys)
    share["Share"] = 100 * share["Papers_Reaching"] / share["Papers_Total"]
    return share


In [3]:
assignments_df = pd.concat(
    [
        assign_prps(pd.read_csv(path)).assign(Year=int(path.parent.name.split("_")[1]))
        for path in sorted(BENCHMARK_ROOT.glob("tpwrs_*/adjudicated_ra_affinities.csv"))
    ],
    ignore_index=True,
)
assignments_df["Affinity_Level"] = assignments_df["Affinity_Level"].astype(LEVEL_DTYPE)
# assignments_df["Era"] = pd.cut(assignments_df["Year"], bins=ERA_BINS, labels=ERA_LABELS)

papers_per_year = assignments_df.groupby("Year")["Abstract_Index"].nunique().rename("Papers_Total")
# papers_per_era = (
#     assignments_df.groupby("Era", observed=True)["Abstract_Index"].nunique().rename("Papers_Total")
# )
YEARS = papers_per_year.index.tolist()
YEAR_TICKTEXT = [
    f"{year}<br><span style='font-size:9px'>{papers_per_year[year]}"
    f"{'*' if year in PARTIAL_YEARS else ''}</span>"
    for year in YEARS
]


print(assignments_df.shape, "rows |", len(YEARS), "years |", papers_per_year.sum(), "papers")
papers_per_year
# papers_per_year.sum()

(36030, 7) rows | 12 years | 6005 papers


Year
2015    358
2016    531
2017    498
2018    703
2019    511
2020    483
2021    559
2022    451
2023    499
2024    602
2025    452
2026    358
Name: Papers_Total, dtype: int64

In [4]:


so_screening_df = pd.read_csv(SO_SCREENING_DIR / "TPWRS_2000_2026_ieee_SO.csv")

screening_lookup = so_screening_df.set_index("doi")[["Has_System_Operator", "Has_Industry_Tech"]]

assignments_df = (
    assignments_df.drop(columns=["Has_System_Operator", "Has_Industry_Tech"], errors="ignore")
    .join(screening_lookup, on="Abstract_Index")
)

assignments_df[["Has_System_Operator", "Has_Industry_Tech"]] = (
    assignments_df[["Has_System_Operator", "Has_Industry_Tech"]]
    .fillna(False)
    .astype(bool)
)
assignments_df["Has_SO_or_Industry"] = (
    assignments_df[["Has_System_Operator", "Has_Industry_Tech"]]
    .any(axis=1)
    .fillna(False)
    .astype(bool)
)

/tmp/ipykernel_8226/1061388299.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [5]:
papers_per_year_so = (
    assignments_df.groupby(["Year", "Has_System_Operator"])["Abstract_Index"]
    .nunique().rename("Papers_Total")
)

reach_share = share_by(assignments_df, "Year", "Has_System_Operator", totals=papers_per_year_so)

all_year_reach = (
    reach_share.groupby(["Threshold", "Has_System_Operator"], as_index=False)[
        ["Papers_Reaching", "Papers_Total"]
    ]
    .sum()
    .assign(Year="all")
)

all_year_reach["Share"] = 100 * all_year_reach["Papers_Reaching"] / all_year_reach["Papers_Total"]
all_year_reach.round(1)


,Threshold,Has_System_Operator,Papers_Reaching,Papers_Total,Year,Share
0,High,False,1155,5826,all,19.8
1,High,True,49,179,all,27.4
2,Moderate+,False,4890,5826,all,83.9
3,Moderate+,True,144,179,all,80.4


In [6]:
assignments_df_with_industry = assignments_df[(assignments_df.Has_System_Operator)]

papers_per_year_with_so = assignments_df_with_industry.groupby("Year")["Abstract_Index"].nunique().rename("Papers_Total")
reach_share_with_so = share_by(assignments_df_with_industry, "Year", totals=papers_per_year_with_so)


In [7]:
industry_counts = pd.DataFrame(
    {"All papers": papers_per_year, "SO affiliated": papers_per_year_with_so}
).fillna(0).astype(int)
industry_counts["Share_%"] = (
    100 * industry_counts["SO affiliated"] / industry_counts["All papers"]
)

counts_fig = px.bar(
    industry_counts.reset_index().melt(
        id_vars=["Year", "Share_%"],
        value_vars=["All papers", "SO affiliated"],
        var_name="Corpus",
        value_name="Papers",
    ),
    x="Year",
    y="Papers",
    color="Corpus",
    barmode="group",
    color_discrete_map={"All papers": CONTEXT, "SO affiliated": ACCENT},
    labels={"Papers": "Papers"},
    title="TPWRS papers per year: full corpus vs SO affiliated",
)
counts_fig.add_scatter(
    x=industry_counts.index,
    y=industry_counts["Share_%"],
    mode="lines+markers",
    name="SO affiliated share",
    line={"color": GAIN, "width": 2},
    marker={"size": 8},
    yaxis="y2",
)
counts_fig.update_layout(
    yaxis2={
        "title": "SO affiliated share (%)",
        "overlaying": "y",
        "side": "right",
        "ticksuffix": "%",
        "rangemode": "tozero",
    }
)
counts_fig.update_xaxes(tickmode="array", tickvals=YEARS)
counts_fig.show()

industry_counts.round(1)

,All papers,SO affiliated,Share_%
Year,,,
2015,358,18,5.0
2016,531,22,4.1
2017,498,17,3.4
2018,703,20,2.8
2019,511,16,3.1
2020,483,15,3.1
2021,559,19,3.4
2022,451,17,3.8
2023,499,7,1.4


In [8]:
from scipy.stats import fisher_exact, norm

Z = norm.ppf(0.975)


def wilson_ci(reaching, total):
    """Wilson score interval for a share, in percentage points."""
    centre = (reaching + Z ** 2 / 2) / (total + Z ** 2)
    half = Z * (reaching * (total - reaching) / total + Z ** 2 / 4) ** 0.5 / (total + Z ** 2)
    return 100 * (centre - half), 100 * (centre + half)


reach_counts = all_year_reach.set_index(["Threshold", "Has_System_Operator"])
rows = []
for threshold in THRESHOLDS:
    reaching_so, total_so = reach_counts.loc[(threshold, True), ["Papers_Reaching", "Papers_Total"]]
    reaching_non, total_non = reach_counts.loc[(threshold, False), ["Papers_Reaching", "Papers_Total"]]
    share_so, share_non = 100 * reaching_so / total_so, 100 * reaching_non / total_non
    low_so, high_so = wilson_ci(reaching_so, total_so)
    low_non, high_non = wilson_ci(reaching_non, total_non)
    difference = share_so - share_non
    difference_low = difference - ((share_so - low_so) ** 2 + (high_non - share_non) ** 2) ** 0.5
    difference_high = difference + ((high_so - share_so) ** 2 + (share_non - low_non) ** 2) ** 0.5
    odds_ratio, p_value = fisher_exact(
        [[reaching_so, total_so - reaching_so], [reaching_non, total_non - reaching_non]]
    )
    rows.append(
        {
            "Threshold": threshold,
            "SO_Papers": total_so,
            "SO_Share": share_so,
            "SO_CI": f"[{low_so:.1f}, {high_so:.1f}]",
            "Non_SO_Papers": total_non,
            "Non_SO_Share": share_non,
            "Non_SO_CI": f"[{low_non:.1f}, {high_non:.1f}]",
            "Difference_pp": difference,
            "Difference_Low_pp": difference_low,
            "Difference_High_pp": difference_high,
            "Difference_CI": f"[{difference_low:+.1f}, {difference_high:+.1f}]",
            "Odds_Ratio": odds_ratio,
            "Fisher_p": p_value,
        }
    )

all_year_test = pd.DataFrame(rows)
all_year_test.round(3)

,Threshold,SO_Papers,SO_Share,SO_CI,Non_SO_Papers,Non_SO_Share,Non_SO_CI,Difference_pp,Difference_Low_pp,Difference_High_pp,Difference_CI,Odds_Ratio,Fisher_p
0,Moderate+,179,80.447,"[74.0, 85.6]",5826,83.934,"[83.0, 84.9]",-3.487,-9.977,1.747,"[-10.0, +1.7]",0.788,0.216
1,High,179,27.374,"[21.4, 34.3]",5826,19.825,"[18.8, 20.9]",7.549,1.454,14.577,"[+1.5, +14.6]",1.524,0.017


In [9]:
PERMUTATION_SEED = 20260824
PERMUTATION_DRAWS = 10_000

paper_reach = (
    assignments_df.assign(
        **{
            threshold: assignments_df["Affinity_Level"].isin(levels)
            for threshold, levels in THRESHOLDS.items()
        }
    )
    .groupby(["Abstract_Index", "Year", "Has_System_Operator"], observed=True)[list(THRESHOLDS)]
    .any()
    .reset_index()
)

# Shuffling the flag within a year keeps each year's paper count and reach rate intact, so the null
# distribution isolates the flag itself rather than the corpus-wide time trend.
is_so = paper_reach["Has_System_Operator"].to_numpy()
year_blocks = [np.flatnonzero(paper_reach["Year"].to_numpy() == year) for year in YEARS]
rng = np.random.default_rng(PERMUTATION_SEED)

rows = []
for threshold in THRESHOLDS:
    reaches = paper_reach[threshold].to_numpy()
    observed = 100 * (reaches[is_so].mean() - reaches[~is_so].mean())
    null_gaps = np.empty(PERMUTATION_DRAWS)
    for draw in range(PERMUTATION_DRAWS):
        shuffled = is_so.copy()
        for block in year_blocks:
            shuffled[block] = rng.permutation(is_so[block])
        null_gaps[draw] = 100 * (reaches[shuffled].mean() - reaches[~shuffled].mean())
    rows.append(
        {
            "Threshold": threshold,
            "Observed_pp": observed,
            "Null_SD_pp": null_gaps.std(),
            "Luck_Band_pp": Z * null_gaps.std(),
            "Beats_Luck": abs(observed) > Z * null_gaps.std(),
            "Permutation_p": (1 + (np.abs(null_gaps) >= abs(observed)).sum()) / (PERMUTATION_DRAWS + 1),
        }
    )

permutation_test = pd.DataFrame(rows)
permutation_test.round(3)

,Threshold,Observed_pp,Null_SD_pp,Luck_Band_pp,Beats_Luck,Permutation_p
0,Moderate+,-3.487,2.792,5.473,False,0.225
1,High,7.549,2.913,5.710,True,0.014


In [10]:
effect_plot = all_year_test.merge(
    permutation_test[['Threshold', 'Permutation_p']],
    on='Threshold',
)
effect_plot['Threshold_Label'] = effect_plot.apply(
    lambda row: (
        f"{row['Threshold']}<br>"
        f"SO {row['SO_Share']:.1f}% (n={row['SO_Papers']:,}) vs "
        f"non-SO {row['Non_SO_Share']:.1f}% (n={row['Non_SO_Papers']:,})"
    ),
    axis=1,
)
effect_plot['Error_Plus'] = effect_plot['Difference_High_pp'] - effect_plot['Difference_pp']
effect_plot['Error_Minus'] = effect_plot['Difference_pp'] - effect_plot['Difference_Low_pp']

effect_fig = px.scatter(
    effect_plot,
    x='Difference_pp',
    y='Threshold_Label',
    error_x='Error_Plus',
    error_x_minus='Error_Minus',
    category_orders={'Threshold_Label': effect_plot['Threshold_Label'].tolist()},
    hover_data={
        'Difference_Low_pp': ':.1f',
        'Difference_High_pp': ':.1f',
        'Permutation_p': ':.3f',
        'Error_Plus': False,
        'Error_Minus': False,
    },
    labels={'Difference_pp': 'Difference in probability (SO minus non-SO, pp)', 'Threshold_Label': ''},
    title='System Operator affiliation and Research-Agenda classification',
)
effect_fig.update_traces(marker=dict(size=11, color=ACCENT), error_x=dict(thickness=1.5, width=5))
effect_fig.add_vline(x=0, line_color=INK_MUTED, line_width=1, line_dash='dot')
effect_fig.update_layout(
    # width=FIGURE_DIM[0],
    # height=260,
    showlegend=False,
    margin=dict(l=200, r=30, t=50, b=55),
)
effect_fig.update_xaxes(ticksuffix=' pp', zeroline=False)
effect_fig.update_yaxes(categoryorder='array', categoryarray=effect_plot['Threshold_Label'].tolist())
effect_fig.show()

In [11]:
headline_fig = px.line(
    reach_share,
    x="Year",
    y="Papers_Reaching",
    color="Threshold",
    facet_col = "Has_System_Operator",
    color_discrete_map={"Moderate+": CONTEXT, "High": ACCENT},
    markers=True,
    line_shape="linear",
    category_orders={"Threshold": list(THRESHOLDS.keys())},
    labels={"Share": "Share of papers (%)"},
)
headline_fig.show()

In [12]:


headline_fig2 = px.line(
    reach_share_with_so,
    x="Year",
    y="Share",
    color="Threshold",
    # facet_col = "Has_System_Operator",
    color_discrete_map={"Moderate+": CONTEXT, "High": ACCENT},
    markers=True,
    line_shape="linear",
    category_orders={"Threshold": list(THRESHOLDS.keys())},
    labels={"Share": "Share of papers (%)"},
)
headline_fig2.show()